# RL data pipeline — staged frame (no ML)

This notebook runs the **data pipeline** exactly the way `just export-data`
does — `build_export_frame` — and stops **before** any ML: no
normalization stats, no environment, no model. You see the raw staged
frame the pipeline hands the training side.

Pipeline stages (in column order):

| stage | contents |
|---|---|
| `timestamp` + OHLCV | ISO-8601 bar time, `open/high/low/close/vwap/volume/count` |
| `signals` | merged exogenous columns (`bid/ask` funding seam here) |
| `features` | `FeaturePipeline.compute` output, pre-transform |
| `warmup` | `True` for indicator look-back rows the agent never trades on |

**Ticker note:** the default config points `funding_features_file` at the
ETH funding file (`configs/default.yaml`), so the default ticker here is
`ETH_USD`. Pick another ticker *and* its own signal file, or set
`signal_require_ticker=False`.


In [ ]:
# Make the repo importable from any kernel (Zed / Jupyter):
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # notebooks/ -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from kraken_trading_bot.rl.export import build_export_frame, write_export_csv

pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 200)
print("ok — repo root:", REPO_ROOT)


In [ ]:
# Run the pipeline for a ticker. `pages` = OHLC pages (~720 bars each,
# Kraken REST cap); `episode_bars` keeps only the trailing N bars.
TICKER = "ETH_USD"        # default config's funding signals are ETH-scoped
PAGES = 3                 # ~2k bars; raise for a longer look
# EPISODE_BARS = 720      # uncomment to trim to the trailing 720 bars

frame = build_export_frame(TICKER, pages=PAGES)
stages = frame.attrs["stages"]

frame.head(3)



In [ ]:
# Run the pipeline for a ticker. `pages` = OHLC pages (~720 bars each,
# Kraken REST cap); `episode_bars` keeps only the trailing N bars.
TICKER = "ETH_USD"        # default config's funding signals are ETH-scoped
PAGES = 3                 # ~2k bars; raise for a longer look
# EPISODE_BARS = 720      # uncomment to trim to the trailing 720 bars

frame = build_export_frame(TICKER, pages=PAGES)
stages = frame.attrs["stages"]

print(f"{TICKER}: {len(frame)} bars x {len(frame.columns)} columns")
for stage, cols in stages.items():
    print(f"  {stage:10s}: {len(cols)} columns")
print("\nFirst rows:")
frame.head(3)


In [ ]:
# The authoritative stage decomposition (frame.attrs["stages"]):
stage_rows = []
for stage, cols in stages.items():
    stage_rows.append({"stage": stage, "n": len(cols), "columns": ", ".join(cols)})
pd.DataFrame(stage_rows)


In [ ]:
# ── Plots — one representative look per stage, no ML anywhere ──
ts = pd.to_datetime(frame["time"], unit="s", utc=True)
warm = frame["warmup"].to_numpy()
warm_idx = np.flatnonzero(warm)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True,
                          gridspec_kw={"height_ratios": [2.2, 1, 1.4, 1]})

# 0 · OHLCV stage — price with high/low band, plus volume twin
ax = axes[0]
ax.plot(ts, frame["close"], lw=1.2, color="#1f77b4", label="close")
ax.fill_between(ts, frame["low"], frame["high"], alpha=0.25, color="#1f77b4",
                label="high/low")
ax.set_ylabel(f"{TICKER.replace('_','/')} price")
ax.legend(loc="upper left", fontsize=9)

# 1 · volume (still OHLCV stage)
ax = axes[1]
ax.bar(ts, frame["volume"], width=0.003, color="#2ca02c", alpha=0.7)
ax.set_ylabel("volume")

# 2 · signals stage — the funding-seam bid/ask spread
ax = axes[2]
if "bid" in frame.columns and "ask" in frame.columns:
    ax.plot(ts, frame["ask"], lw=1.0, color="#d62728", label="ask")
    ax.plot(ts, frame["bid"], lw=1.0, color="#9467bd", label="bid")
    ax.set_ylabel("bid/ask")
    ax.legend(loc="upper left", fontsize=9)
else:
    ax.text(0.5, 0.5, "no signal columns merged", ha="center", va="center",
            transform=ax.transAxes)
    ax.set_yticks([])

# 3 · one feature per feature group the config actually produced
ax = axes[3]
feature_cols = stages["features"]
groups = {
    "price": [c for c in feature_cols if c.startswith("return_")][:1],
    "technical": [c for c in feature_cols if c.startswith("rsi_")][:1],
    "volume": [c for c in feature_cols if c.startswith("volume_")][:1],
    "signals": [c for c in feature_cols if c == "funding_rate" or c == "spread"][:1],
}
for label, cols in groups.items():
    if cols:
        ax.plot(ts, frame[cols[0]], lw=1.0, label=f"{label}: {cols[0]}")
ax.set_ylabel("features")
ax.legend(loc="upper left", fontsize=9)

# Warmup shading on every panel
for ax in axes:
    if warm_idx.size:
        ax.axvspan(ts.iloc[0], ts.iloc[warm_idx[-1]], color="gray", alpha=0.15)
ax.set_xlabel("UTC")

# Annotation for the warmup region only once
if warm_idx.size:
    axes[0].text(ts.iloc[warm_idx[-1]], axes[0].get_ylim()[1],
                 " warmup", fontsize=9, color="gray", va="top")

fig.suptitle(f"{TICKER} — data pipeline staged frame ({len(frame)} bars, "
             f"no ML)", y=0.995)
fig.tight_layout()
plt.show()


In [ ]:
# Optionally write the same staged frame back out (mirrors `just export-data`):
# CSV loses stages; the JSON sidecar keeps the column-block split.
csv_path = REPO_ROOT / "exports" / f"{TICKER}.csv"
# write_export_csv(frame, csv_path)   # uncomment to (re)write
print(f"would write: {csv_path} ({len(frame)} rows)")


## How to read this

- **Top panel** — the raw OHLCV stage: price with the high/low envelope,
  and volume underneath.
- **bid/ask** — a merged *signal* column pair from the funding seam
  (`signals` stage); the `spread` feature derived from them is in the
  features panel.
- **Features** — a single representative column per feature group
  (`price`, `technical`, `volume`, `signals`). All 60 are in `frame`;
  see the stage table above for the full list.
- **Gray band** — the `warmup` region: indicator look-back rows that are
  in the frame but that the RL environment would never trade on.

Change `TICKER` or `PAGES` and re-run the two cells above. Every cell is
the plain pipeline — there is no RL here.
